# Lexsi concept #1 — triage baseline (the "before" run) — Kaggle variant

Attach `lexsi-pack.zip` as a Kaggle dataset input (or upload it in the notebook), enable a GPU accelerator, then run top to bottom. Same harness, same checkpoint, same held-out set as the Colab run.

## 1. Get the pack into the session
Upload a zip of the `lexsi/` folder (triage-tasks.jsonl + baseline/) when prompted — or replace this cell with a `git clone` once the submission repo exists.

In [ ]:
import zipfile, glob, os, shutil
zips = glob.glob('/kaggle/input/**/*.zip', recursive=True)
zips = sorted(zips, key=lambda p: ('lexsi' not in p.lower(), p))  # prefer the lexsi pack if several zips are attached
assert zips, 'lexsi pack zip not found under /kaggle/input — attach it as a dataset input'
zipfile.ZipFile(zips[0]).extractall('/kaggle/working')
base = [p for p in glob.glob('/kaggle/working/**/baseline', recursive=True)][0]
os.chdir(base)
print('working dir:', os.getcwd())


## 2. Install + prove the scorer before any GPU is spent

In [ ]:
!pip -q install -r requirements.txt
!python3 test_scoring.py

## 3. Authenticate to Hugging Face (gated checkpoint — REQUIRED)
aya-expanse-8b is gated: without a login this run 401s (learned live Nov 13). This cell pulls the `HF_TOKEN` secret (Kaggle: Add-ons > Secrets) and logs in. It is the cell the Nov 13 live session had to improvise — now it ships in the pack.

In [ ]:
import os
tok = os.environ.get('HF_TOKEN')
if not tok:
    try:
        from kaggle_secrets import UserSecretsClient
        tok = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception as e:
        print('kaggle secret read failed:', type(e).__name__)
assert tok, 'HF_TOKEN not available — add it as a Kaggle secret named HF_TOKEN (Add-ons > Secrets)'
os.environ['HF_TOKEN'] = tok
from huggingface_hub import login, HfApi
login(token=tok)
print('HF user:', HfApi(token=tok).whoami().get('name'))


## 4. Resolve the checkpoint
Decision (Nov 13): the baseline runs on **CohereLabs/aya-expanse-8b** — its gated access is ACCEPTED on the Snedz HF account and the lexsi-baseline token is ready, and the first authenticated run already started on it, so before/after consistency pins this checkpoint. `CohereLabs/tiny-aya-global` (correct id; the pack originally tried the nonexistent `CohereLabs/tiny-aya`) stays as fallback only — its own gated-licence accept is still outstanding, so do not switch to it mid-entry.

In [ ]:
from huggingface_hub import HfApi
api = HfApi()
MODEL = None
for cand in ['CohereLabs/aya-expanse-8b', 'CohereLabs/tiny-aya-global']:
    try:
        api.model_info(cand)
        MODEL = cand
        break
    except Exception as e:
        print(cand, '->', type(e).__name__)
print('MODEL =', MODEL)
assert MODEL, 'no Cohere checkpoint resolved — check the id on huggingface.co'


## 5. The baseline run (greedy decoding, the held-out set, no training)

T4 route: `--load-in-8bit` is REQUIRED — aya-expanse-8b at FP16 needs ~17.9 GB and does not fit a 16 GB T4 (it spills to CPU RAM and crawls instead of failing). 8-bit weights are ~8 GB; the load precision is fingerprinted in `scores.json`.

In [ ]:
import subprocess, sys
out = 'results/' + MODEL.split('/')[-1] + '-base'
r = subprocess.run([sys.executable, 'run_baseline.py', '--backend', 'transformers', '--model', MODEL, '--load-in-8bit', '--out', out])
print('exit', r.returncode)


## 6. Read the before numbers

In [ ]:
print(open(out + '/summary.md').read())

## 7. Formalize into the AuditKIT record
First inspect the real agent_eval API (its signatures were UNVERIFIED when this pack was built), then wire per `auditkit_config.yaml`.

In [ ]:
import subprocess, sys
# Verified path (Nov 13, auditkit 1.1.11): formalize_auditkit.py wraps this run
# into the AuditKIT record; verdicts matched scoring.py 104/104 on the echo smoke.
r = subprocess.run([sys.executable, 'formalize_auditkit.py', '--run', out,
                    '--out', out + '/auditkit'])
print('exit', r.returncode)
# Fallback inspection if the API moved: import auditkit.agent_eval as ae; print(dir(ae))


## 8. Download the results (scores + predictions + traces are submission artefacts)

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/lexsi-baseline-results', 'zip', out)
print('results archive: /kaggle/working/lexsi-baseline-results.zip (download it from the notebook Output panel)')
